# Assignment 1: Lumen Beauty and BakeBox

Part 1 uses the Week 2 files and cleaning. Part 2 uses the Week 3 mining steps. The CSV files are in this folder. Run the notebook from the top.


In [1]:
import warnings
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings("ignore", category=DeprecationWarning)


## Part 1. Lumen Beauty, September 2026

S07 is in the export twice. I keep one row, so the month is 12 sessions.


In [2]:
sessions = pd.read_csv("livestream_sessions.csv")
df = sessions.drop_duplicates().reset_index(drop=True)
print(len(sessions), "rows in the file,", len(df), "sessions kept")


13 rows in the file, 12 sessions kept


## Task 1.1. Two ways to rank a stream

Compute GMV per viewer for each of the 12 sessions and rank them. Is the top session by GMV per viewer also the session with the highest peak viewership? Explain what each ranking measures, why they differ, and which one the founder should use when deciding what a good stream means.

GMV per viewer = session GMV / peak viewers.


In [3]:
df["gmv_per_viewer"] = df["gmv"] / df["viewers_peak"]
cols = ["session_id", "viewers_peak", "gmv", "gmv_per_viewer"]
print(df.sort_values("gmv_per_viewer", ascending=False)[cols].round(2).to_string(index=False))


session_id  viewers_peak      gmv  gmv_per_viewer
       S05           379  6852.76           18.08
       S12          1427 25728.68           18.03
       S09          1305 22936.99           17.58
       S03           844 14783.91           17.52
       S01          1419 22832.04           16.09
       S06          1331 21178.30           15.91
       S04          1249 19137.11           15.32
       S08          1135 17245.42           15.19
       S10           523  7220.19           13.81
       S02          1207 16496.93           13.67
       S07           346  4409.22           12.74
       S11           344  4085.66           11.88


S05 is first on GMV per viewer (18.08 CNY) but the room was only 379 people and the stream made 6,852.76 CNY. S12 has the most viewers (1,427) and almost the same rate (18.03 CNY), with the month's highest sales, 25,728.68 CNY. Peak viewers says how many people showed up. GMV per viewer says what each of them was worth in sales. Use GMV per viewer, and call it a good stream only when the audience is large too. That stream is S12.


## Task 1.2. Month two

Compute each SKU's sales growth from week 1 to week 4 and name the two fastest-growing SKUs. For each of the two, recommend one concrete action for the store's second month, and tie the action to a number from the data.

Growth = week-4 GMV / week-1 GMV. These are store sales, not stream sales.


In [4]:
w = pd.read_csv("livestream_weekly_sales.csv")
w["growth"] = w["gmv_w4"] / w["gmv_w1"]
w["growth_pct"] = (w["growth"] - 1) * 100
show = w.sort_values("growth", ascending=False)
print(show[["sku", "product_name", "price", "gmv_w1", "gmv_w4", "growth", "growth_pct"]].round(3).to_string(index=False))
for _, row in show.head(2).iterrows():
    print(f"{row['product_name']}: about {row['gmv_w4'] / row['price']:.0f} units in week 4 at the list price")


sku       product_name  price   gmv_w1   gmv_w4  growth  growth_pct
P02    Cloud Cream 50g    199 15418.24 30296.71   1.965      96.499
P03   Mist Toner 150ml     89  2466.52  4544.28   1.842      84.239
P05        Lip Tint 04     69  1601.11  2866.81   1.791      79.051
P06       Eye Gel 15ml    129  4207.01  7277.29   1.730      72.980
P04     Clay Mask 100g     79  2817.02  4366.44   1.550      55.002
P01   Hydra Serum 30ml    169  5922.82  9072.81   1.532      53.184
P07     Sun Milk SPF50     99  3815.54  5775.08   1.514      51.357
P08 Cleansing Balm 90g    109  5103.71  7417.19   1.453      45.329
Cloud Cream 50g: about 152 units in week 4 at the list price
Mist Toner 150ml: about 51 units in week 4 at the list price


Cloud Cream grew from 15,418.24 to 30,296.71 CNY (96.5%). Week 4 is about 152 units at 199 CNY. Order about 150 units a week and put it first on every stream.

Mist Toner grew from 2,466.52 to 4,544.28 CNY (84.2%). Week 4 is about 51 units at 89 CNY, still a small share of the store. Give it one pitch per stream and keep about 50 units a week.


## Task 1.3. Did the streams pay for themselves?

Streaming costs about 800 CNY per hour (host, venue, samples). Estimate the total streaming cost for September and compare it with the GMV the streams generated. State your conclusion, then name the single assumption your conclusion is most sensitive to, and show how the conclusion moves if that assumption changes.

The 800 CNY does not include the cost of the products. GMV is what was sold, not what the store keeps.


In [5]:
hours = df["duration_min"].sum() / 60
cost = 800 * hours
stream_gmv = df["gmv"].sum()
print(f"{df['duration_min'].sum():.0f} minutes, {hours:.1f} hours")
print(f"Cost: {cost:,.0f} CNY")
print(f"Stream GMV: {stream_gmv:,.2f} CNY")
print(f"Break-even margin: {cost / stream_gmv:.2%}")
for margin in (0.30, 0.08):
    kept = margin * stream_gmv
    print(f"{margin:.0%} margin keeps {kept:,.0f} CNY")
print(f"At 1,600 CNY/hour the bill is {1600 * hours:,.0f} CNY")


1380 minutes, 23.0 hours
Cost: 18,400 CNY
Stream GMV: 182,907.21 CNY
Break-even margin: 10.06%
30% margin keeps 54,872 CNY
8% margin keeps 14,633 CNY
At 1,600 CNY/hour the bill is 36,800 CNY


The bill is 18,400 CNY. The streams sold 182,907.21 CNY. They cover the bill if the store keeps more than 10.06% of that GMV. At a 30% margin it keeps about 54,872 CNY. At 8% it keeps about 14,633 CNY and the bill is not covered. That margin is the assumption the answer stands on. Doubling the hourly cost to 1,600 CNY (bill 36,800) still works at 30% and still fails at 8%.


## Part 2. BakeBox, August 2026

One row in the file is one product on an order. A basket is the set of products on that order. Mining is the Week 3 setup: minimum support 0.05, no confidence filter, rules ranked by lift.


In [6]:
NAME = {
    "BB01": "bread_flour", "BB02": "instant_yeast", "BB03": "cake_flour",
    "BB04": "matcha_powder", "BB05": "cocoa_powder", "BB06": "sugar",
    "BB07": "butter", "BB08": "bread_improver", "BB09": "cake_mold",
    "BB10": "bamboo_whisk", "BB11": "piping_bag", "BB12": "nozzle_set",
    "BB13": "spatula_set", "BB14": "mixing_bowl", "BB15": "kitchen_scale",
    "BB16": "oven_thermometer", "BB17": "baking_tray", "BB18": "parchment_roll",
    "BB19": "cupcake_cases", "BB20": "doilies", "BB21": "gift_boxes",
    "BB22": "sprinkle_mix", "BB23": "food_coloring", "BB24": "cake_toppers",
    "BB25": "apron",
}

raw = pd.read_csv("order_baskets.csv")
raw["item"] = raw["sku"].map(NAME)
orders = raw.groupby("order_id").agg(
    channel=("channel", "first"),
    items=("item", lambda s: sorted(set(s))),
)
price = raw.groupby("item")["unit_price"].first()
print(len(orders), "orders")
print(orders["channel"].value_counts().to_string())

def rule_text(r):
    left = " + ".join(sorted(r["antecedents"]))
    right = " + ".join(sorted(r["consequents"]))
    return left + " -> " + right

def rank_rules(rules):
    view = rules[["rule", "support", "confidence", "lift"]].copy()
    view["lift_shown"] = view["lift"].round(3)
    view["left_items"] = rules["antecedents"].apply(len)
    view["left_price"] = rules["antecedents"].apply(lambda a: float(price[list(a)].sum()))
    view = view.sort_values(["lift_shown", "left_items", "left_price"], ascending=[False, True, True])
    return view[["rule", "support", "confidence", "lift"]].reset_index(drop=True)

def mine(sub, min_support=0.05):
    te = TransactionEncoder()
    onehot = pd.DataFrame(
        te.fit(sub["items"]).transform(sub["items"]),
        columns=te.columns_,
        index=sub.index,
    )
    frequent = apriori(onehot, min_support=min_support, use_colnames=True)
    rules = association_rules(frequent, metric="confidence", min_threshold=0.0)
    rules["rule"] = rules.apply(rule_text, axis=1)
    return onehot, rules


2800 orders
channel
storefront    1984
livestream     816


## Task 2.1. The strongest rules

Mine association rules on all baskets with min_support = 0.05. Report the three rules with the highest lift, with support, confidence and lift for each. Pick one rule and explain its business meaning in a short paragraph: what should the merchant do with it?


In [7]:
onehot, rules = mine(orders)
ranked = rank_rules(rules)
print(ranked.head(3).round(3).to_string(index=False))


                              rule  support  confidence   lift
apron -> mixing_bowl + spatula_set    0.079       0.804 10.091
mixing_bowl + spatula_set -> apron    0.079       0.991 10.091
mixing_bowl -> apron + spatula_set    0.079       0.759  9.622


The top three rules are the apron, the mixing bowl, and the spatula set. Use mixing bowl + spatula set -> apron: support 0.079, confidence 0.991, lift 10.091. When the bowl and the spatula are already in the basket, the apron is bought 99.1% of the time, in about 221 orders. Sell the three as one bundle, and offer the missing piece when the cart already has two.


## Task 2.2. The bestseller that goes with nothing

Paper doilies are the single most bought product in the store. Using your own mining output, show that they form no strong rule with any other product, and explain why high support alone does not make an item a good cross-selling anchor. If association is not what justifies keeping such an item, what should the keep-or-delist decision rest on?

Most doily pairs sit under the 5% line, so the lift is counted directly, as in Week 3.


In [8]:
print(f"doilies support: {onehot['doilies'].mean():.3f} ({int(onehot['doilies'].sum())} orders)")
rows = []
for col in onehot.columns:
    if col == "doilies":
        continue
    both = int((onehot["doilies"] & onehot[col]).sum())
    lift = (both / onehot["doilies"].sum()) / onehot[col].mean()
    rows.append((col, both, round(float(lift), 3)))
decoy = pd.DataFrame(rows, columns=["item", "orders_together", "lift_with_doilies"])
decoy = decoy.sort_values("lift_with_doilies").reset_index(drop=True)
print(f"lift from {decoy['lift_with_doilies'].min():.3f} to {decoy['lift_with_doilies'].max():.3f}")
print(f"doilies rules at min_support 0.05: {rules['rule'].str.contains('doilies').sum()}")
print(decoy.to_string(index=False))


doilies support: 0.280 (784 orders)
lift from 0.804 to 1.250
doilies rules at min_support 0.05: 0
            item  orders_together  lift_with_doilies
           sugar               45              0.804
     baking_tray               40              0.893
       cake_mold              112              0.952
   matcha_powder               82              0.957
    bamboo_whisk               79              0.970
   food_coloring               44              0.982
   kitchen_scale               44              0.982
  bread_improver               78              0.984
    cocoa_powder               47              0.987
     bread_flour              116              0.998
           apron               77              1.000
  parchment_roll              133              1.002
   instant_yeast              117              1.004
    sprinkle_mix               45              1.004
    cake_toppers               45              1.004
      cake_flour              119              1.010
 

Doilies are in 784 of 2,800 orders (support 0.280). Lift with every other product is between 0.804 and 1.250, and no doilies rule clears support 0.05. Being common does not mean it pulls another product. Keep doilies if the item itself makes money. Drop them if it does not.


## Task 2.3. One store, two channels

Re-run the mining on the livestream orders and on the storefront orders separately. Find one rule that is strong in the livestream slice but absent from the storefront slice at the same threshold, and explain what creates such a difference. Then answer the operations question: should the merchant read one blended rule set or one rule set per channel, and what does your answer change for the storefront bundle page and for the live show?


In [9]:
live = orders[orders["channel"] == "livestream"]
store = orders[orders["channel"] == "storefront"]
_, rules_live = mine(live)
_, rules_store = mine(store)
print(len(live), "livestream orders,", len(rules_live), "rules")
print(len(store), "storefront orders,", len(rules_store), "rules")

def show_rule(frame, text):
    hit = frame[frame["rule"] == text]
    if hit.empty:
        print(text + ": ABSENT")
    else:
        print(hit[["rule", "support", "confidence", "lift"]].round(3).to_string(index=False))

print("Livestream")
show_rule(rules_live, "spatula_set -> apron")
print("Storefront")
show_rule(rules_store, "spatula_set -> apron")
print(rank_rules(rules_store).head(4).round(3).to_string(index=False))


816 livestream orders, 72 rules
1984 storefront orders, 32 rules
Livestream
                rule  support  confidence  lift
spatula_set -> apron    0.271         1.0 3.692
Storefront
spatula_set -> apron: ABSENT
                         rule  support  confidence  lift
     piping_bag -> nozzle_set    0.109       1.000 9.185
     nozzle_set -> piping_bag    0.109       1.000 9.185
bamboo_whisk -> matcha_powder    0.100       1.000 8.371
matcha_powder -> bamboo_whisk    0.100       0.835 8.371


Spatula set -> apron is strong on the livestream (support 0.271, confidence 1.000, lift 3.692) and absent on the storefront. The host sells that pair on the show. Storefront shoppers do not pick it on their own. Read one rule set per channel. Keep the apron set in the live script. On the storefront bundle page, use pairs from that channel, such as piping bag -> nozzle set (support 0.109, lift 9.185) and bamboo whisk -> matcha powder (support 0.100, lift 8.371).


## AI-use appendix

I did half of this assignment myself. Grok helped me draft the findings and the parts of the write-up I did not finish on my own. The figures come from the cells above.

S09 has an average watch time of 252 on a stream that lasted 150 minutes. I left that number as it was exported. Nothing in Tasks 1.1 to 1.3 uses watch time, so the sales, viewer, and cost figures are unchanged either way.
